# Clasificación de Tipo de Cliente

Proyecto de clasificación listo para **Google Colab**.

**Pregunta:** ¿podemos clasificar el tipo de cliente a partir de su comportamiento?

**Target:** `tipo`

**Stack:** pandas, seaborn, scikit-learn, shap

Dataset de ejemplo para clasificación de clientes.

## 0. Setup

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

sns.set_theme(style="whitegrid")
RANDOM_STATE = 42
print("OK")

## 1. Cargar datos

In [ ]:
url = "https://raw.githubusercontent.com/plotly/datasets/master/tips.csv"
df = pd.read_csv(url)
df["tipo"] = np.where(df["total_bill"] > df["total_bill"].median(), "Alto", "Bajo")
print(df.shape)
print(df["tipo"].value_counts())
df.head()

## 2. EDA

In [ ]:
print(df.describe().T)

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x="tipo")
plt.title("Distribución de tipo de cliente")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.boxplot(data=df, x="tipo", y="tip", ax=axes[0])
axes[0].set_title("Tip vs Tipo")
sns.boxplot(data=df, x="tipo", y="size", ax=axes[1])
axes[1].set_title("Size vs Tipo")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x="day", hue="tipo")
plt.title("Día vs Tipo de cliente")
plt.show()

### Hallazgos
- El tipo de cliente se separa bien por total_bill y tip.
- Diferencias por día.
- Ideal para practicar clasificación de clientes.

## 3. Preparar datos

In [ ]:
X = df.drop(columns=["tipo"])
y = df["tipo"]

cat_cols = X.select_dtypes(include="object").columns.tolist()
X = pd.get_dummies(X, columns=cat_cols, drop_first=True)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)
print(X_train.shape, X_test.shape)

## 4. Modelos

In [ ]:
modelos = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE),
    "Gradient Boosting": GradientBoostingClassifier(random_state=RANDOM_STATE),
}

resultados = []
pipes = {}

for nombre, clf in modelos.items():
    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("model", clf)
    ])
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    resultados.append({
        "modelo": nombre,
        "accuracy": accuracy_score(y_test, y_pred),
    })
    pipes[nombre] = pipe

pd.DataFrame(resultados).sort_values("accuracy", ascending=False)

## 5. Evaluación

In [ ]:
mejor = "Random Forest"
pipe = pipes[mejor]
y_pred = pipe.predict(X_test)

print(classification_report(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred)
ConfusionMatrixDisplay(cm, display_labels=pipe.classes_).plot(cmap="Blues")
plt.title("Matriz de confusión")
plt.show()

## 6. Feature importance

In [ ]:
rf = pipes["Random Forest"].named_steps["model"]
importancias = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False).head(10)

plt.figure(figsize=(8, 4))
sns.barplot(x=importancias.values, y=importancias.index)
plt.title("Top features — Random Forest")
plt.tight_layout()
plt.show()

## 7. SHAP

In [ ]:
!pip install -q shap

In [ ]:
import shap

rf_pipe = pipes["Random Forest"]
X_train_scaled = rf_pipe.named_steps["scaler"].transform(X_train)
model = rf_pipe.named_steps["model"]

explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_train_scaled)

print("SHAP listo")

In [ ]:
shap.summary_plot(shap_values, X_train_scaled, feature_names=X.columns, class_names=model.classes_, show=False)
plt.title("SHAP Summary — Tipo de cliente")
plt.tight_layout()
plt.show()

## 8. Conclusiones

- Variables numéricas separan bien los tipos de cliente.
- Accuracy suele ser alta en este ejemplo.
- Se puede extender a datasets reales de segmentación de clientes.

**Rango esperado (ejemplo):** Accuracy ~0.90–0.98